### Setup


In [1]:
# Add the parent directory of the current working directory to the Python path at runtime. 
# In order to import modules from the src directory.
import os
import sys 

current_dir = os.getcwd()
parent_dir = os.path.dirname(current_dir)
sys.path.insert(0, parent_dir)

In [2]:
import numpy as np
import pandas as pd
import bambi as bmb
import arviz as az

from prettytable import PrettyTable

from src.stat_utils import *
from src.anl_utils import load_data, get_session_data

### Load and prepare data

In [3]:
sim_results_folder = '../results/simulation'
data_folder = '../data'
sync_at_file = os.path.join(sim_results_folder, 'first_session_arnold_tongues.npy')
emp_at_file = os.path.join(data_folder, 'Experiment.csv')

In [4]:
def fit_transform(df, columns, sessions):
    """Z-score transform specified columns within specified sessions.
    
    Parameters:
    df (pd.DataFrame): The input DataFrame.
    columns (list): List of column names to be transformed.
    sessions (list): List of session identifiers (int) to consider for mean and std calculation.
    
    """
    transformed_df = df.copy()
    session_mask = df['SessionID'].isin(sessions)
    for column in columns:
        mean = df.loc[session_mask, column].mean()
        std = df.loc[session_mask, column].std()
        transformed_df[column] = (transformed_df[column] - mean) / std
    return transformed_df



In [ ]:
# Load the simulations results and the empirical data
sync_results = np.load(sync_at_file)
sync_results_vector = sync_results.mean(axis=0).flatten()
data = load_data(emp_at_file)

# Dummy code session 9
data_transfer = data.copy()
data_transfer['Transfer'] = (data_transfer['SessionID'] == 9)

# Filter data for learning phase (Sessions 1 to 8)
data_learning = data[data['SessionID'] <= 8].copy()

# Map synchrony values to each condition in DataFrame
data_learning['Synchrony'] = data_learning['Condition'].apply(lambda x: sync_results_vector[x-1])

# Z-score the relevant columns
data_learning = zscore_data(data_learning, ['ContrastHeterogeneity', 'GridCoarseness', 'Synchrony'])
data_transfer = fit_transform(data_transfer, ['ContrastHeterogeneity', 'GridCoarseness'], sessions=[1,2,3,4,5,6,7,8])

# Center the session number
mean_session = data_learning['SessionID'].mean()
data_learning['SessionCentered'] = data_learning['SessionID'] - mean_session
data_transfer['SessionCentered'] = data_transfer['SessionID'] - mean_session

### Define statistical models

In [ ]:
model_features = bmb.Model(
    "Correct ~ 1 + ContrastHeterogeneity * GridCoarseness + SessionCentered * (ContrastHeterogeneity + GridCoarseness) +  (1 + SessionCentered + ContrastHeterogeneity * GridCoarseness|SubjectID)",
    data=data_learning,
    family="bernoulli"
)

Does learning occur (i.e., does session have an effect on performance)? 

Do the effects of contrast heterogeneity and/or grid coarseness depend on session?

In [7]:
idata_features = model_features.fit(
    draws=2000, tune=2000, target_accept=0.9,
    idata_kwargs={"log_likelihood": True}, progressbar=False
)

Modeling the probability that Correct==1
Initializing NUTS using jitter+adapt_diag...
/home/mario/miniconda3/envs/bat_env/lib/python3.13/site-packages/pytensor/link/c/cmodule.py:2968: UserWarning: PyTensor could not link to a BLAS installation. Operations that might benefit from BLAS will be severely degraded.
This usually happens when PyTensor is installed via pip. We recommend it be installed via conda/mamba/pixi instead.
Alternatively, you can use an experimental backend such as Numba or JAX that perform their own BLAS optimizations, by setting `pytensor.config.mode == 'NUMBA'` or passing `mode='NUMBA'` when compiling a PyTensor function.
For more options and details see https://pytensor.readthedocs.io/en/latest/troubleshooting.html#how-do-i-configure-test-my-blas-library
  warnings.warn(
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [Intercept, ContrastHeterogeneity, GridCoarseness, ContrastHeterogeneity:GridCoarseness, SessionCentered, SessionCentered:ContrastHeterogeneity, Ses

In [8]:
predictors = ["SessionCentered", "ContrastHeterogeneity", "GridCoarseness", "SessionCentered:ContrastHeterogeneity","SessionCentered:GridCoarseness", "ContrastHeterogeneity:GridCoarseness"]
directions = ['greater', 'less', 'less','less','less', 'greater']

posterior = posterior_table(idata_features, predictors, directions)

odds_ratios = OR_table(idata_features, predictors)

print("One-sided posterior probabilities:")
print(posterior)
print("\nOdds ratios:")
print(odds_ratios)

az.summary(idata_features, var_names=predictors, hdi_prob=0.95)

One-sided posterior probabilities:
+---------------------------------------+-----------+-------+
|               Predictor               | direction |   P   |
+---------------------------------------+-----------+-------+
|            SessionCentered            |  greater  | 1.000 |
|         ContrastHeterogeneity         |    less   | 1.000 |
|             GridCoarseness            |    less   | 1.000 |
| SessionCentered:ContrastHeterogeneity |    less   | 1.000 |
|     SessionCentered:GridCoarseness    |    less   | 0.954 |
|  ContrastHeterogeneity:GridCoarseness |  greater  | 1.000 |
+---------------------------------------+-----------+-------+

Odds ratios:
+---------------------------------------+-------+--------------+---------------+
|               Predictor               |  Mean | Lower (2.5%) | Upper (97.5%) |
+---------------------------------------+-------+--------------+---------------+
|            SessionCentered            | 1.100 |    1.067     |     1.133     |
|      

,mean,sd,hdi_2.5%,hdi_97.5%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
SessionCentered,0.095,0.015,0.066,0.126,0.000,0.000,3424.0,3962.0,1.0
ContrastHeterogeneity,-0.943,0.132,-1.191,-0.654,0.003,0.003,2477.0,2690.0,1.0
GridCoarseness,-0.316,0.029,-0.377,-0.262,0.000,0.000,4143.0,4406.0,1.0
SessionCentered:ContrastHeterogeneity,-0.081,0.005,-0.090,-0.070,0.000,0.000,10639.0,5628.0,1.0
SessionCentered:GridCoarseness,-0.008,0.005,-0.018,0.001,0.000,0.000,9082.0,6466.0,1.0
ContrastHeterogeneity:GridCoarseness,0.272,0.034,0.201,0.342,0.001,0.001,3680.0,4457.0,1.0


In [19]:
import numpy as np
import arviz as az
from scipy.special import expit

# names must match your fitted model with SessionCentered
b_int = az.extract(idata_features, var_names=["SessionCentered:GridCoarseness"]).to_dataframe().iloc[:, -1].to_numpy()
b_int = np.asarray(b_int)

b0 = az.extract(idata_features, var_names=["Intercept"]).to_dataframe().iloc[:, -1].to_numpy()
b0 = np.asarray(b0)

# Probability change for a 1-session increase at GC = +1 SD, CH=0, SessC baseline (0)
p0 = expit(b0)
p1 = expit(b0 + b_int)  # add only the interaction contribution (ΔSession=1, ΔGC=1)
delta_p = p1 - p0

# Summaries
h = az.hdi(delta_p, hdi_prob=0.95)
out = {
    "mean_Δp": float(delta_p.mean()),
    "HDI_95%_low": float(h[0]),
    "HDI_95%_high": float(h[1]),
    "P(Δp>0)": float((delta_p > 0).mean()),
    "P(|Δp|<0.02)": float((np.abs(delta_p) < 0.02).mean())
}
print(out)


{'mean_Δp': -0.0014714308179523473, 'HDI_95%_low': -0.0031860585716049217, 'HDI_95%_high': 0.0002513775922442507, 'P(Δp>0)': 0.04625, 'P(|Δp|<0.02)': 1.0}


In [20]:
def session_simple_effects_CH(idata, sessions, hdi=0.95):
    # Extract posterior draws for CH main and CH:Session interaction
    posterior = az.extract(idata, var_names=["ContrastHeterogeneity",
                                        "SessionCentered:ContrastHeterogeneity"]).to_dataframe()
    beta_contrast_heterogeneity   = posterior["ContrastHeterogeneity"].to_numpy()
    beta_session_ch_interaction = posterior["SessionCentered:ContrastHeterogeneity"].to_numpy()

    rows = []
    for session in sessions:
        beta_draws = beta_contrast_heterogeneity + beta_session_ch_interaction  * session
        odds_ratio_draws   = np.exp(beta_draws)

        hdi_low, hdi_high = az.hdi(beta_draws, hdi_prob=hdi)
        odds_ratio_low, odds_ratio_high = az.hdi(odds_ratio_draws, hdi_prob=hdi)

        rows.append({
            "Session": session,
            "beta_mean": float(beta_draws.mean()),
            f"beta_hdi_{int((1-hdi)/2*100)}%": float(hdi_low),
            f"beta_hdi_{int((1+hdi)/2*100)}%": float(hdi_high),
            "OR_mean": float(odds_ratio_draws.mean()),
            f"OR_hdi_{int((1-hdi)/2*100)}%": float(odds_ratio_low),
            f"OR_hdi_{int((1+hdi)/2*100)}%": float(odds_ratio_high),
            "Pr_beta_less_0": float((beta_draws < 0).mean())
        })
    return pd.DataFrame(rows)

    
sessions = np.arange(1, 9) - 4.5
tbl_ch_by_session = session_simple_effects_CH(idata_features, sessions, hdi=0.95)
print(tbl_ch_by_session)


   Session  beta_mean  beta_hdi_2%  beta_hdi_97%   OR_mean  OR_hdi_2%  \
0     -3.5  -0.660230    -0.908597     -0.372313  0.521329   0.381072   
1     -2.5  -0.740897    -0.989575     -0.455388  0.480881   0.351458   
2     -1.5  -0.821565    -1.078034     -0.543169  0.443584   0.322911   
3     -0.5  -0.902233    -1.163039     -0.627158  0.409192   0.299585   
4      0.5  -0.982901    -1.236380     -0.698535  0.377477   0.277744   
5      1.5  -1.063569    -1.331587     -0.793826  0.348230   0.255209   
6      2.5  -1.144236    -1.419139     -0.882014  0.321258   0.241922   
7      3.5  -1.224904    -1.495404     -0.954064  0.296384   0.217807   

   OR_hdi_97%  Pr_beta_less_0  
0    0.662440             1.0  
1    0.610086             1.0  
2    0.560603             1.0  
3    0.518263             1.0  
4    0.479112             1.0  
5    0.441148             1.0  
6    0.413948             1.0  
7    0.377008             1.0  


### Transfer session analysis

In [10]:
model_transfer = bmb.Model(
    "Correct ~ 1 + ContrastHeterogeneity * GridCoarseness + Transfer + SessionCentered * (ContrastHeterogeneity + GridCoarseness) +  (1 + SessionCentered + ContrastHeterogeneity * GridCoarseness|SubjectID)",
    data=data_transfer,
    family="bernoulli"
)

In [11]:
idata_transfer = model_transfer.fit(
    draws=2000, tune=2000, target_accept=0.9,
    idata_kwargs={"log_likelihood": True}, progressbar=False
)

Modeling the probability that Correct==1
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [Intercept, ContrastHeterogeneity, GridCoarseness, ContrastHeterogeneity:GridCoarseness, Transfer, SessionCentered, SessionCentered:ContrastHeterogeneity, SessionCentered:GridCoarseness, 1|SubjectID_sigma, 1|SubjectID_offset, SessionCentered|SubjectID_sigma, SessionCentered|SubjectID_offset, ContrastHeterogeneity|SubjectID_sigma, ContrastHeterogeneity|SubjectID_offset, GridCoarseness|SubjectID_sigma, GridCoarseness|SubjectID_offset, ContrastHeterogeneity:GridCoarseness|SubjectID_sigma, ContrastHeterogeneity:GridCoarseness|SubjectID_offset]
Sampling 4 chains for 2_000 tune and 2_000 draw iterations (8_000 + 8_000 draws total) took 3760 seconds.
There were 10 divergences after tuning. Increase `target_accept` or reparameterize.


In [12]:
predictors = ["SessionCentered", "ContrastHeterogeneity", "GridCoarseness", "SessionCentered:ContrastHeterogeneity","SessionCentered:GridCoarseness", "ContrastHeterogeneity:GridCoarseness", "Transfer"]

az.summary(idata_transfer, var_names=predictors, hdi_prob=0.95)

,mean,sd,hdi_2.5%,hdi_97.5%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
SessionCentered,0.079,0.008,0.062,0.094,0.000,0.000,4786.0,5126.0,1.0
ContrastHeterogeneity,-0.898,0.129,-1.138,-0.638,0.003,0.002,2379.0,3455.0,1.0
GridCoarseness,-0.305,0.031,-0.369,-0.243,0.001,0.001,3481.0,4132.0,1.0
SessionCentered:ContrastHeterogeneity,-0.047,0.004,-0.055,-0.038,0.000,0.000,8859.0,5676.0,1.0
SessionCentered:GridCoarseness,-0.001,0.004,-0.009,0.007,0.000,0.000,10556.0,5718.0,1.0
ContrastHeterogeneity:GridCoarseness,0.266,0.033,0.201,0.334,0.001,0.001,3842.0,4372.0,1.0
Transfer,-0.430,0.040,-0.508,-0.353,0.000,0.000,9227.0,5671.0,1.0


In [13]:
def post_mean_prob_pop(model, idata, test_data):
    """
    Population-level posterior of mean accuracy for the given test_data.
    Uses kind="mean" (expected probability) and excludes group-specific effects.
    Returns: array of shape (n_draws,) with the mean probability per draw.
    """
    pp = model.predict(
        idata=idata,
        data=test_data,
        kind="mean",
        include_group_specific=False,  # <- population level: no REs
        inplace=False
    )
    # Bambi stores this in the posterior group with a var like "<response>_mean".
    # Grab the first *_mean variable defensively.
    mean_vars = [v for v in pp.posterior.data_vars if v.endswith("_mean")]
    if not mean_vars:
        # fallback: some versions use "y_mean"
        mean_vars = [v for v in pp.posterior.data_vars if v == "y_mean"]
    varname = mean_vars[0]

    arr = pp.posterior[varname].values  # shape: (chains, draws, obs)
    draws = arr.shape[0] * arr.shape[1]
    arr2 = arr.reshape(draws, arr.shape[2])  # (draws, obs)
    return arr2.mean(axis=1)  # average across obs -> one mean prob per draw

In [14]:
# Posterior distributions of mean accuracy (population level)

data_s9 = get_session_data(data_transfer, 9)
data_s2 = get_session_data(data_transfer, 2)

posterior_s9 = post_mean_prob_pop(model_transfer, idata_transfer, data_s9)
posterior_s2 = post_mean_prob_pop(model_transfer, idata_transfer, data_s2)

# Contrasts & summaries
def summarize_diff(a, b, hdi=0.95, rope=0.02):
    d = a - b
    h = az.hdi(d, hdi_prob=hdi)
    return {
        "P(a<b)": float((a < b).mean()),
        "mean_diff": float(d.mean()),
        "hdi_low": float(h[0]),
        "hdi_high": float(h[1]),
        "P_within_ROPE(±{:.0%})".format(rope): float((np.abs(d) < rope).mean())
    }

print("S9 vs S2:", summarize_diff(posterior_s9, posterior_s2))

S9 vs S2: {'P(a<b)': 0.381, 'mean_diff': 0.0028808222713845727, 'hdi_low': -0.019129798568172984, 'hdi_high': 0.022707916940192363, 'P_within_ROPE(±2%)': 0.935375}
